In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from model import ScalingLaw, GetAsymptVar, SampleAlpha
from model_general import ScalingLaw as ScalingLawGeneral
from constants import lower_bounds, test_models, delete_models, Y_names_tidy, Y_names, B, lrs, scheduler_factors, reps, n_epochs, random_seed
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ThreadPoolExecutor, as_completed

K = 4
eps = 1e-3
Y_names = Y_names[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Data

In [2]:
# Loading
df = pd.merge(pd.read_csv('data/df_full_v1.csv').drop('Unnamed: 0', axis=1),
              pd.read_csv('data/df_full_v2.csv').drop('Unnamed: 0', axis=1), 
              on=['model', 'family', 'size', 'tokens', 'flops'], how='outer')

# Creating data objects
Y = np.array(df.loc[:,Y_names])
Y = np.clip(Y, a_min=eps, a_max=1-eps)
        
X = np.log(np.array(df.loc[:,['size','tokens']]))
X = np.hstack((X,(X[:,0]*X[:,1])[:,None]))

D = np.array(pd.get_dummies(df.family)).astype(int)
I = np.ones(shape=(D.shape[0],1))
C = np.array([lower_bounds[s] for s in Y_names]).reshape((1,-1))

Training models

In [3]:
test_models.keys()

dict_keys(['meta-llama-3', 'qwen2', 'yi-1.5', 'olmo', 'smollm', 'gemma2'])

In [ ]:
models = {}

mu = np.mean(X, axis=0, keepdims=True)
std = np.std(X, axis=0, keepdims=True)

X_train = (X-mu)/std
Y_train = Y
D_train = D

# Model 1
models['model'] = ScalingLaw(K)
models['model'].fit(X = X_train,
                      Y = Y_train,
                      D = D_train,
                      C = C,
                      B = B,
                      lrs = lrs,
                      scheduler_factors = scheduler_factors,
                      reps = reps,
                      n_epochs = n_epochs,    
                      verbose = True,
                      device = device)

# Model 2
models['model-general'] = ScalingLawGeneral(K)
models['model-general'].fit(X_mean = X_train,
                              Y = Y_train,
                              D = D_train,
                              C = C,
                              X_phi = np.zeros(X_train.shape),
                              B = B,
                              lrs = lrs,
                              scheduler_factors = scheduler_factors,
                              reps = reps,
                              n_epochs = n_epochs,    
                              verbose = True,
                              device = device)
    
np.save(f"models/prediction_intervals/comparing_models.npy", models)

Different lrs:   0%|          | 0/3 [00:00<?, ?it/s]

Different scheduler factors:   0%|          | 0/2 [00:00<?, ?it/s]

Reps:   0%|          | 0/5 [00:00<?, ?it/s]

Training ME model:   0%|          | 0/20000 [00:00<?, ?it/s]

Results (checking if the fit is the same or close enough)

In [ ]:
models = np.load("models/comparing_models.npy", allow_pickle=True).item()